# 14 — BTC S2 no-regime null: continuous HAR-GARCH-t

**Purpose.** Module 13 found exactly one primary candidate that passed the pre-null sign gate: BTC, block-6, S2, 30m. This notebook asks whether a continuous, non-switching volatility process can manufacture an HMM advantage at least as strong.

The null simulates **5-minute returns**, then rebuilds the exact Module 13 block-6 emission and reruns the same candidate gate. It does not simulate HMM states.

Null DGP:

\[
\log b_t = X_t\beta,\qquad
q_t=\omega+\alpha u_{t-1}^2+\beta_g q_{t-1},\qquad
r_t=\sqrt{b_tq_t}\,z_t,
\]

with causal HAR + clock predictors in \(X_t\), continuous GARCH(1,1) residual variance, and standardized Student-t shocks. The real timestamp grid, missing-return mask, exact-zero-return mask, weekend/calendar pattern and deterministic clock are preserved.

### Primary statistic

BTC S2 had to beat four non-switching benchmarks in two score views. Therefore the primary scalar is the **weakest** of those eight mean-session QLIKE gains:

\[
T_{min}=\min_j \Delta QLIKE_j.
\]

The candidate is unusual under the null only if null paths rarely achieve a \(T_{min}\) at least as large as the real value. This is a **candidate-level** null, not yet a family-wise correction over every asset/model inspected in Module 13.


In [1]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 220)
pd.set_option('display.width', 260)
pd.set_option('display.max_rows', 1000)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found from current working directory.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling import vol_hmm as vhm
from data_engineering.modeling import vol_hmm_null as vhn

DEV_START, SCREEN_TRAIN_END, DEV_END = '2026-09-22', '2026-09-28', '2026-10-07'
N_PATHS = int(os.getenv('M14_PATHS', '200'))
N_INIT = int(os.getenv('M14_N_INIT', '4'))
MAX_ITER = int(os.getenv('M14_MAX_ITER', '150'))
SIM_SEED = int(os.getenv('M14_SEED', '14000'))
NULL_SCOPE = os.getenv('M14_NULL_SCOPE', 'all').lower()  # all | train
RESUME = os.getenv('M14_RESUME', '1') == '1'
CHECKPOINT_EVERY = int(os.getenv('M14_CHECKPOINT_EVERY', '10'))
OUT = PROJECT_ROOT / 'notebooks' / 'HSMM'
PATHS_CSV = OUT / '14_btc_s2_null_paths.csv'
SUMMARY_CSV = OUT / '14_btc_s2_null_summary.csv'
REAL_GATE_CSV = OUT / '14_btc_s2_real_gate.csv'
FIT_JSON = OUT / '14_btc_s2_null_fit.json'

if vhm.VOL_HMM_VERSION < '0.2.0' or vhn.VOL_HMM_NULL_VERSION < '0.1.0':
    raise RuntimeError('Old Module 13/14 code: git pull and restart the kernel.')
if NULL_SCOPE not in {'all', 'train'}:
    raise ValueError('M14_NULL_SCOPE must be all or train')
print(f'vol_hmm {vhm.VOL_HMM_VERSION} | null {vhn.VOL_HMM_NULL_VERSION} | paths {N_PATHS} | scope {NULL_SCOPE}')


vol_hmm 0.2.0 | null 0.1.0 | paths 200 | scope all


## 1. Load the frozen development window

Only BTC is needed because it is the sole Module 13 candidate that earned the no-regime test.


In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)
cols = set(con.execute('DESCRIBE time_series_feature_state').df()['column_name'])
required = {'state_id','state_time','underlying','return_5m','futures_index_price','is_weekend','in_settlement_window','option_atm_quote_ok','option_iv_atm_forward'}
missing = sorted(required - cols)
if missing:
    raise RuntimeError(f'time_series_feature_state lacks {missing}')
DEV_LAST_BAR = pd.Timestamp(f'{DEV_END} 08:00', tz='UTC')
want = sorted(required)
g = con.execute(
    f"SELECT {', '.join(want)} FROM time_series_feature_state "
    f"WHERE underlying='BTC' AND state_time <= TIMESTAMPTZ '{DEV_LAST_BAR.isoformat()}' ORDER BY state_time"
).df()
con.close()

g['state_time'] = pd.to_datetime(g['state_time'], utc=True)
g['session_date'] = ri.session_key(g['state_time'])
keep = ((g['session_date'] >= DEV_START) & (g['session_date'] <= DEV_END)) | (g['state_time'] == DEV_LAST_BAR)
g = g[keep].reset_index(drop=True)
DEV_SESSIONS = sorted(s for s in g['session_date'].unique() if DEV_START <= s <= DEV_END)
SCREEN_TRAIN = {s for s in DEV_SESSIONS if s <= SCREEN_TRAIN_END}
SCREEN_EVAL = {s for s in DEV_SESSIONS if s > SCREEN_TRAIN_END}
NULL_FIT_SESSIONS = set(DEV_SESSIONS) if NULL_SCOPE == 'all' else set(SCREEN_TRAIN)
print(f'BTC rows {len(g):,} | dev {len(DEV_SESSIONS)} | train {len(SCREEN_TRAIN)} | eval {len(SCREEN_EVAL)} | null-fit {len(NULL_FIT_SESSIONS)}')


BTC rows 4,608 | dev 16 | train 7 | eval 9 | null-fit 16


## 2. Reproduce the real Module 13 candidate before simulating anything

This cell is a guard against code drift. The recomputed eight BTC S2 gains must match the committed Module 13 CSV. If they do not, the null test aborts rather than testing a different statistic.


In [3]:
real_stats, real_gate = vhn.evaluate_btc_s2_gate(
    g, SCREEN_TRAIN, SCREEN_EVAL, DEV_SESSIONS,
    seed_base=1300, n_init=N_INIT, max_iter=MAX_ITER,
)
real_gate.to_csv(REAL_GATE_CSV, index=False)

m13 = pd.read_csv(OUT / '13_screening_comparisons.csv')
expected = m13[(m13.underlying == 'BTC') & (m13.horizon == '30m') & (m13.block == 6) &
               (m13.model == 'S2@b6') &
               (m13['vs'].isin(['STATIC_K1@b6','ADAPT_K1@b6','M12_M1','M12_M1_ADAPT']))].copy()
expected_map = {(r['vs'], r['view']): float(r['mean_session_diff']) for _, r in expected.iterrows()}
recomputed_map = {(r['vs'], r['view']): float(r['gain']) for _, r in real_gate.iterrows()}
if set(expected_map) != set(recomputed_map):
    raise RuntimeError('Module 14 real-gate keys do not match Module 13 CSV.')
max_abs_diff = max(abs(expected_map[k] - recomputed_map[k]) for k in expected_map)
print('max abs reproduction error:', max_abs_diff)
if max_abs_diff > 1e-6:
    raise RuntimeError('Module 14 does not reproduce Module 13 BTC S2 gate; stop before null simulation.')

display(real_gate.round(6))
print('\nReal candidate statistics:')
display(pd.DataFrame([real_stats]).T.rename(columns={0:'value'}).round(6))


max abs reproduction error: 2.220446049250313e-16


,model,vs,view,gain,sessions_won,n_rows,n_sessions
0,S2@b6,STATIC_K1@b6,all,1.535925,0.444444,1726,9
1,S2@b6,STATIC_K1@b6,non_overlapping,0.268913,0.666667,290,9
2,S2@b6,ADAPT_K1@b6,all,3.576424,0.555556,1726,9
3,S2@b6,ADAPT_K1@b6,non_overlapping,0.631816,0.666667,290,9
4,S2@b6,M12_M1,all,0.707413,0.444444,1726,9
5,S2@b6,M12_M1,non_overlapping,0.093273,0.666667,290,9
6,S2@b6,M12_M1_ADAPT,all,2.566558,0.444444,1726,9
7,S2@b6,M12_M1_ADAPT,non_overlapping,0.417057,0.666667,290,9



Real candidate statistics:


,value
gain__STATIC_K1@b6__all,1.535925
won__STATIC_K1@b6__all,0.444444
gain__STATIC_K1@b6__non_overlapping,0.268913
won__STATIC_K1@b6__non_overlapping,0.666667
gain__ADAPT_K1@b6__all,3.576424
won__ADAPT_K1@b6__all,0.555556
gain__ADAPT_K1@b6__non_overlapping,0.631816
won__ADAPT_K1@b6__non_overlapping,0.666667
gain__M12_M1__all,0.707413
won__M12_M1__all,0.444444


## 3. Fit the continuous no-regime DGP

Default `M14_NULL_SCOPE=all` fits the null DGP on all development sessions. That does **not** alter the Module 13 train/evaluation split inside each simulated path. It gives the no-regime process the best chance to learn the continuous persistence and volatility drift seen anywhere in the development sample, making the null conservative.


In [4]:
null_fit = vhn.fit_continuous_null(g, NULL_FIT_SESSIONS)
FIT_JSON.write_text(json.dumps(null_fit.to_dict(), indent=2))
display(pd.DataFrame([{
    'n_fit': null_fit.n_fit,
    'garch_alpha': null_fit.garch_alpha,
    'garch_beta': null_fit.garch_beta,
    'garch_omega': null_fit.garch_omega,
    'alpha_plus_beta': null_fit.garch_alpha + null_fit.garch_beta,
    'nu': null_fit.nu,
    'smear': null_fit.smear,
    'base_scale': null_fit.base_scale,
    'median_base_var': null_fit.median_base_var,
}]).round(6))


,n_fit,garch_alpha,garch_beta,garch_omega,alpha_plus_beta,nu,smear,base_scale,median_base_var
0,3699,0.067142,0.17696,0.755898,0.244102,5.0,4.970216,1.000001,0.000001


## 4. Parametric bootstrap

Every path simulates returns from the continuous null and then reruns the exact BTC S2 gate. A checkpoint CSV is written periodically and can be resumed with `M14_RESUME=1`.


In [5]:
if RESUME and PATHS_CSV.exists():
    paths = pd.read_csv(PATHS_CSV)
    done = set(pd.to_numeric(paths['path'], errors='coerce').dropna().astype(int))
    print(f'resuming with {len(done)} completed paths')
else:
    paths = pd.DataFrame()
    done = set()

records = paths.to_dict('records') if len(paths) else []
t0 = time.perf_counter()
for b in range(N_PATHS):
    if b in done:
        continue
    rec = {'path': b, 'sim_seed': SIM_SEED + b, 'valid': False, 'error': ''}
    try:
        r_star = vhn.simulate_returns(g, null_fit, SIM_SEED + b, preserve_zero_mask=True)
        g_star = vhn.replace_returns(g, r_star)
        st, _ = vhn.evaluate_btc_s2_gate(
            g_star, SCREEN_TRAIN, SCREEN_EVAL, DEV_SESSIONS,
            seed_base=1300, n_init=N_INIT, max_iter=MAX_ITER,
        )
        rec.update(st)
        rec['valid'] = True
    except Exception as exc:
        rec['error'] = f'{type(exc).__name__}: {exc}'
    records.append(rec)

    if (b + 1) % CHECKPOINT_EVERY == 0 or b + 1 == N_PATHS:
        paths = pd.DataFrame(records).sort_values('path').reset_index(drop=True)
        paths.to_csv(PATHS_CSV, index=False)
        good = int(paths['valid'].fillna(False).astype(bool).sum()) if 'valid' in paths else 0
        print(f'checkpoint {b + 1}/{N_PATHS} | valid {good}')

paths = pd.DataFrame(records).sort_values('path').reset_index(drop=True)
paths.to_csv(PATHS_CSV, index=False)
display(paths.tail())


checkpoint 10/200 | valid 10
checkpoint 20/200 | valid 20
checkpoint 30/200 | valid 30
checkpoint 40/200 | valid 40
checkpoint 50/200 | valid 50
checkpoint 60/200 | valid 60
checkpoint 70/200 | valid 70
checkpoint 80/200 | valid 80
checkpoint 90/200 | valid 90
checkpoint 100/200 | valid 100
checkpoint 110/200 | valid 110
checkpoint 120/200 | valid 120
checkpoint 130/200 | valid 130
checkpoint 140/200 | valid 140
checkpoint 150/200 | valid 150
checkpoint 160/200 | valid 160
checkpoint 170/200 | valid 170
checkpoint 180/200 | valid 180
checkpoint 190/200 | valid 190
checkpoint 200/200 | valid 200


,path,sim_seed,valid,error,gain__STATIC_K1@b6__all,won__STATIC_K1@b6__all,gain__STATIC_K1@b6__non_overlapping,won__STATIC_K1@b6__non_overlapping,gain__ADAPT_K1@b6__all,won__ADAPT_K1@b6__all,gain__ADAPT_K1@b6__non_overlapping,won__ADAPT_K1@b6__non_overlapping,gain__M12_M1__all,won__M12_M1__all,gain__M12_M1__non_overlapping,won__M12_M1__non_overlapping,gain__M12_M1_ADAPT__all,won__M12_M1_ADAPT__all,gain__M12_M1_ADAPT__non_overlapping,won__M12_M1_ADAPT__non_overlapping,min_gain,gate_pass,degenerate_fit,intercept_gap,min_occupancy,max_occupancy,sigma_ratio,max_p_stay,min_p_stay,max_floor_share,train_loglik_gain_x2
195,195,14195,True,,-1.001477,0.444444,0.236317,0.555556,-8.115381,0.222222,-1.010533,0.333333,-1.027947,0.333333,0.286889,0.444444,-8.373932,0.222222,-1.005070,0.222222,-8.373932,False,False,1.144804,0.123765,0.876235,1.384940,0.978331,0.813085,0.013231,1.820086
196,196,14196,True,,-3.503129,0.222222,-0.654120,0.222222,-4.950071,0.222222,-0.870064,0.333333,-3.006274,0.222222,-0.516609,0.111111,-4.633165,0.222222,-0.778145,0.222222,-4.950071,False,False,1.021263,0.317471,0.682529,1.134925,0.864605,0.778226,0.016844,13.856266
197,197,14197,True,,11.821684,0.555556,2.805772,0.555556,3.630719,0.555556,1.159646,0.555556,18.517185,0.444444,3.953724,0.444444,11.299739,0.333333,2.408890,0.555556,1.159646,False,True,2.388748,0.015518,0.984482,1.466156,1.000000,0.755488,0.012094,-17.142132
198,198,14198,True,,-1.942599,0.444444,-0.279427,0.333333,-1.273161,0.444444,-0.244888,0.555556,-2.504998,0.333333,-0.389640,0.333333,-2.744202,0.333333,-0.465407,0.555556,-2.744202,False,False,1.249513,0.068490,0.931510,1.336798,0.977425,0.718165,0.012650,30.987343
199,199,14199,True,,-1.333508,0.444444,-0.059148,0.555556,-3.391327,0.333333,-0.428303,0.444444,-2.527100,0.444444,-0.245353,0.555556,-4.333007,0.333333,-0.577354,0.333333,-4.333007,False,False,1.112411,0.305239,0.694761,1.254446,0.864333,0.747883,0.035477,-24.719121


## 5. Null result

The primary empirical p-value is

\[
p=\frac{1+\#\{T^{sim}_{min}\ge T^{real}_{min}\}}{B+1}.
\]

`gate_pass_rate` is the fraction of continuous-null paths that pass the same eight-sign, non-degenerate gate at all. State separation statistics are secondary diagnostics only.


In [6]:
valid = paths[paths['valid'].fillna(False).astype(bool)].copy()
failure_rate = 1.0 - len(valid) / max(N_PATHS, 1)
if len(valid) < max(50, int(0.90 * N_PATHS)):
    raise RuntimeError(f'Too many failed null paths: {len(valid)}/{N_PATHS} valid. Do not interpret the null.')

p_min = vhn.empirical_p(real_stats['min_gain'], valid['min_gain'].to_numpy(), 'greater')
p_gap = vhn.empirical_p(real_stats['intercept_gap'], valid['intercept_gap'].to_numpy(), 'greater')
p_stay = vhn.empirical_p(real_stats['max_p_stay'], valid['max_p_stay'].to_numpy(), 'greater')
p_lr = vhn.empirical_p(real_stats['train_loglik_gain_x2'], valid['train_loglik_gain_x2'].to_numpy(), 'greater')
gate_pass_rate = float(valid['gate_pass'].astype(bool).mean())
joint_rate = float(((valid['min_gain'] >= real_stats['min_gain']) &
                    (valid['intercept_gap'] >= real_stats['intercept_gap']) &
                    (~valid['degenerate_fit'].astype(bool))).mean())

summary = pd.DataFrame([{
    'n_requested': N_PATHS,
    'n_valid': len(valid),
    'failure_rate': failure_rate,
    'real_min_gain': real_stats['min_gain'],
    'p_min_gain': p_min,
    'null_gate_pass_rate': gate_pass_rate,
    'real_intercept_gap': real_stats['intercept_gap'],
    'p_intercept_gap': p_gap,
    'real_max_p_stay': real_stats['max_p_stay'],
    'p_max_p_stay': p_stay,
    'real_train_lr_x2': real_stats['train_loglik_gain_x2'],
    'p_train_lr_x2': p_lr,
    'joint_min_gain_and_gap_rate': joint_rate,
}])
summary.to_csv(SUMMARY_CSV, index=False)
display(summary.T.rename(columns={0:'value'}).round(6))

q = valid[['min_gain','intercept_gap','max_p_stay','train_loglik_gain_x2']].quantile([0.05,0.5,0.95,0.99])
print('\nNull quantiles:')
display(q.round(6))


,value
n_requested,200.000000
n_valid,200.000000
failure_rate,0.000000
real_min_gain,0.093273
p_min_gain,0.029851
null_gate_pass_rate,0.030000
real_intercept_gap,1.069512
p_intercept_gap,0.353234
real_max_p_stay,0.982531
p_max_p_stay,0.288557



Null quantiles:


,min_gain,intercept_gap,max_p_stay,train_loglik_gain_x2
0.05,-15.291751,0.530335,0.789176,-27.192310
0.50,-4.915078,0.998367,0.949746,18.778500
0.95,-0.147511,1.882018,1.000000,67.973457
0.99,0.209458,2.744604,1.000000,79.447672


## 6. Pre-declared interpretation gate

This module is intentionally stricter than “the real HMM has two separated states.” State separation is common even under continuous volatility.

- **Fail / stop switching for now:** `p_min_gain > 0.10`, or the continuous null itself passes the eight-sign gate frequently.
- **Inconclusive:** `0.05 < p_min_gain <= 0.10`. Do not proceed to HSMM; accumulate more sessions.
- **Candidate survives this conditional null:** `p_min_gain <= 0.05` **and** `null_gate_pass_rate <= 0.05`. This still does not prove a regime. Before HSMM, review selection adjustment across the broader Module 13 search and the block-6 phase-chain architecture.

The intercept-gap, persistence and pseudo-likelihood statistics are diagnostics; they do not override the forecast-based primary test.


In [7]:
if p_min <= 0.05 and gate_pass_rate <= 0.05:
    verdict = 'SURVIVES candidate-level null — next audit selection adjustment + expanding S2/phase architecture before HSMM.'
elif p_min <= 0.10:
    verdict = 'INCONCLUSIVE — do not proceed to HSMM; more independent sessions are needed.'
else:
    verdict = 'FAILS no-regime null — continuous volatility can explain the apparent BTC switching gain; stop switching branch for now.'
print(verdict)


SURVIVES candidate-level null — next audit selection adjustment + expanding S2/phase architecture before HSMM.
